# CS 521 HW4, Problem 2: Certified Training for Multiple Norms (CURE)

Open this notebook in Google Colab and select a GPU runtime (*Runtime > Change runtime type > T4 GPU*).

Complete each part that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the verifier, the evaluation or the default hyperparameters.
Change them only if the homework tells you to.

**Threat model.** Let $f$ be the network and $x$ an image with the label $y$. The network is *certified* at $x$
for the norm $p$ if the verifier proves this: $f_y(x') - f_j(x') > 0$ for all $j \neq y$ and all $x'$ with
$\|x' - x\|_p \leq \epsilon_p$. For $\ell_\infty$, $x'$ must also be in $[0, 1]^{784}$.
We use $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$.
The *union certified accuracy* is the percentage of test images where the network is certified for the two norms.

In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

EPS = {"linf": 0.1, "l2": 1.0}     # the radii of the threat model
NORMS = ["linf", "l2"]

device: cpu


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [ ]:
def load_mnist(train):
    ds = torchvision.datasets.MNIST("data", train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)  # (n, 1, 28, 28)
    return x.to(device), ds.targets.to(device)


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]

torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [ ]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = x.clone(), torch.full((len(x),), -float("inf"), device=x.device)
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = (x + delta)[better], loss[better]
    return best_x.detach()

# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [ ]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = (c - r).clamp(0, 1), (c + r).clamp(0, 1)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
    else:
        mid, rad = layer(c), r * layer.weight.norm(dim=1)
    return mid - rad, mid + rad


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)  # (batch, 10, 10): row j is e_y - e_j


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
    l, u = F.relu(l), F.relu(u)
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)  # (batch, 10, n): W_y - W_j
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = (u + l) / 2, (u - l) / 2
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = l >= 0, (l < 0) & (u > 0)
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | (unstable & (u > -l)), torch.ones_like(l), torch.zeros_like(l))
    return alpha, s, c


def _backward_lower(A, layers, bounds, c0, r, norm):
    # Lower bound of A z, where z is the output of layers[-1] and bounds[i] are the bounds of the output of layers[i].
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = A.clamp(min=0), A.clamp(max=0)
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = (c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1)
        return const + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2) + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[:k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[:k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y))
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0

# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [ ]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(f"{r['name']:<26}" + "  ".join(f"{k} {v:6.2f}" for k, v in r.items()
                                     if k != "time" and isinstance(v, float)) +
              (f"  time/epoch {r['time']:5.1f} s" if "time" in r else ""))

# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [ ]:
WARMUP, RAMP = 1, 10
LR = 4e-3


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s")
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)

# (a) Single-norm certified training

CURE uses the SABR loss from the lectures for each norm. SABR finds an adversarial example $x^*$ with PGD, and then it
calculates the IBP loss on a small ball of radius $\tau = \lambda \epsilon$ around $x^*$. PGD searches for $x^*$ in the
ball of radius $\epsilon - \tau$ around $x$, so that the small ball is inside the full ball.

1. `propagation_region`: Find a center $x^*$ with `pgd` (5 steps) in the ball of radius $\epsilon - \tau$ around $x$.
   Attack the model in eval mode. Then set the model back to train mode.
2. `certified_loss`: Calculate the cross-entropy of $\hat{z} = -\underline{m}$. Here, $\underline{m}$ are the IBP lower
   bounds of the margins on the small ball. Thus $\hat{z}_y = 0$.
   With `reduction="none"`, return the loss of each image.
3. `single_norm_loss`: For $t < 1$, return the standard loss. After that, return `certified_loss` with the radius
   `eps_scale(t) * EPS[norm]`.

The default ratios are `LAM = {"linf": 0.7, "l2": 0.7}`.

In [ ]:
LAM = {"linf": 0.7, "l2": 0.7}


def propagation_region(model, x, y, eps, lam, norm):
    tau = lam * eps
    if eps - tau <= 0:
        return x, tau
    model.eval()
    center = pgd(model, x, y, eps - tau, norm, steps=5)
    model.train()
    return center, tau


def certified_loss(model, x, y, eps, lam, norm, reduction="mean"):
    c, tau = propagation_region(model, x, y, eps, lam, norm)
    return F.cross_entropy(-ibp_margin_bounds(model, c, tau, norm, y), y, reduction=reduction)


def single_norm_loss(model, x, y, t, norm):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    return certified_loss(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)

In [ ]:
def mean_row(rows, name):
    out = {"name": name}
    for k, v in rows[0].items():
        if isinstance(v, float): out[k] = sum(r[k] for r in rows) / len(rows)
    return out

def run(name, loss_fn, seeds=(0,), **kw):
    rows = []
    for s in seeds:
        torch.manual_seed(s); m = make_model()
        sec = train(m, loss_fn, seed=s, verbose=False, **kw)
        rows.append(dict(evaluate(m, name, attacks=True), time=sec))
    return mean_row(rows, name)

results = [run("standard", standard_loss),
           run("linf", lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf")),
           run("l2", lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2"))]
print_results(results)

standard                  clean  97.23  linf   0.00  l2   0.00  union   0.00  pgd_linf   1.85  pgd_l2  12.27  pgd_union   1.82  time/epoch   0.2 s
linf                      clean  93.83  linf  80.33  l2   6.94  union   6.94  pgd_linf  84.15  pgd_l2  69.30  pgd_union  69.11  time/epoch   1.9 s
l2                        clean  92.64  linf  54.33  l2  58.31  union  50.65  pgd_linf  68.42  pgd_l2  78.00  pgd_union  67.98  time/epoch   1.8 s


**(a)** (i) Only partly: the $\ell_\infty$ network is 6.9% $\ell_2$-certified despite 69.3% $\ell_2$ PGD accuracy. The $\ell_2$ network transfers better (54.3% $\ell_\infty$-certified vs 68.4% PGD).

(ii) Standard: both. $\ell_\infty$ network: $\ell_2$. $\ell_2$ network: $\ell_\infty$.

# (b) CURE

Let $\hat{z}^\infty$ and $\hat{z}^2$ be the negative IBP margin bounds on the two small balls of an image.
These are the same values as in `certified_loss`. Let $d^p = \mathrm{softmax}(\hat{z}^p)$. Complete `cure_loss`:
$$\mathcal{L}_{CURE} = \frac{1}{n} \sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right)
+ \eta \cdot \frac{1}{|\gamma|} \sum_{i \in \gamma} \mathrm{KL}\left(d^2_i \,\|\, d^\infty_i\right),$$
Here, $\gamma$ is the set of images in the batch that are certified for $\ell_2$ on their small ball.
For these images, all entries of $\hat{z}^2_i$ except $\hat{z}^2_{i, y_i}$ are negative.
The first term is CURE-Max. The second term is the bound alignment (BA).
For $t < 1$, return the standard loss. If $\gamma$ is empty, the second term is 0.

Then complete `gradient_projection` (GP). Let $g_c$ be the gradient of `cure_loss`, and $g_n$ the gradient of the
cross-entropy loss on the clean images. For each parameter tensor:
$$g = \beta \cdot \mathrm{GP}(g_n, g_c) + (1 - \beta) \cdot g_c, \qquad
\mathrm{GP}(g_n, g_c) = \begin{cases} \cos(g_n, g_c) \cdot g_n & \text{if } \cos(g_n, g_c) > 0 \\ 0 & \text{otherwise.} \end{cases}$$
This is the same operation as in RAMP.

This is a simplified version of CURE. The paper is different in three ways:

* It propagates an $\ell_\infty$ box also for $\ell_2$.
* It uses a different $\ell_2$ attack.
* It applies GP to the updates of a full epoch.

In [ ]:
def zhat(model, x, y, eps, norm):
    # Negative IBP margin bounds on the SABR small ball (z_hat_y = 0).
    c, tau = propagation_region(model, x, y, eps, LAM[norm], norm)
    return -ibp_margin_bounds(model, c, tau, norm, y)


def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    z = {p: zhat(model, x, y, eps_scale(t) * EPS[p], p) for p in NORMS}
    ce = {p: F.cross_entropy(z[p], y, reduction="none") for p in NORMS}
    loss = torch.maximum(ce["linf"], ce["l2"]).mean()
    if eta > 0:
        z2 = z["l2"].scatter(1, y[:, None], -float("inf"))
        gamma = z2.max(1).values < 0                       # l2-certified on the small ball
        if gamma.any():
            log_d2, log_dinf = F.log_softmax(z["l2"][gamma], 1), F.log_softmax(z["linf"][gamma], 1)
            kl = (log_d2.exp() * (log_d2 - log_dinf)).sum(1).mean()
            loss = loss + eta * kl
    return loss


def gradient_projection(g_n, g_c, beta):
    cos = (g_n * g_c).sum() / (g_n.norm() * g_c.norm()).clamp(min=1e-12)
    gp = cos * g_n if cos > 0 else torch.zeros_like(g_n)
    return beta * gp + (1 - beta) * g_c

In [ ]:
for name, loss_fn, beta in [("cure_max", lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), None),
                            ("cure_max_ba", cure_loss, None),
                            ("cure_max_ba_gp", cure_loss, 0.5)]:
    results.append(run(name, loss_fn, seeds=(0, 1), gp_beta=beta))
print_results(results)  # rows for CURE are the mean of seeds 0 and 1

standard                  clean  97.23  linf   0.00  l2   0.00  union   0.00  pgd_linf   1.85  pgd_l2  12.27  pgd_union   1.82  time/epoch   0.2 s
linf                      clean  93.83  linf  80.33  l2   6.94  union   6.94  pgd_linf  84.15  pgd_l2  69.30  pgd_union  69.11  time/epoch   1.9 s
l2                        clean  92.64  linf  54.33  l2  58.31  union  50.65  pgd_linf  68.42  pgd_l2  78.00  pgd_union  67.98  time/epoch   1.8 s
cure_max                  clean  92.19  linf  67.48  l2  57.20  union  57.07  pgd_linf  76.56  pgd_l2  78.08  pgd_union  75.97  time/epoch   3.6 s
cure_max_ba               clean  92.11  linf  66.82  l2  56.94  union  56.83  pgd_linf  76.31  pgd_l2  77.92  pgd_union  75.81  time/epoch   3.7 s
cure_max_ba_gp            clean  92.43  linf  66.72  l2  56.33  union  56.22  pgd_linf  76.42  pgd_l2  77.97  pgd_union  75.89  time/epoch   3.8 s


**(b)** (i) CURE-Max has the highest union certified accuracy (57.1%), and the standard network has the highest clean accuracy (97.2%).

(ii) No (56.8% with BA, 56.2% with BA + GP). BA only acts on images that are already $\ell_2$-certified, but $\ell_2$ is the norm that limits the union. GP trades a little certified accuracy for clean accuracy.

(iii) About 2x, because each batch runs PGD and IBP for both norms.

# (c) Your method

Write the loss of your method in `my_loss`. Train it with `train` and the default settings, as for the other methods.
If your method needs a different update, write your own copy of `train`.
Do not change the verifier or `evaluate`.

In [ ]:
# Our method: CURE-Max, but the certified loss uses the tighter of the IBP and CROWN-IBP bounds.
def ibp_preact_bounds(model, c, r, norm):
    # IBP bounds of every hidden pre-activation.
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    bounds = [(l, u)]
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        bounds.append((mid - rad, mid + rad))
        l, u = bounds[-1]
    return bounds


def crown_ibp_margin_bounds(model, c, r, norm, y):
    # One CROWN backward pass from the margin spec, with IBP intermediate bounds (CROWN-IBP).
    return _backward_lower(spec_matrix(y), linear_layers(model), ibp_preact_bounds(model, c, r, norm), c, r, norm)


def zhat_tight(model, x, y, eps, norm):
    c, tau = propagation_region(model, x, y, eps, LAM[norm], norm)
    return -torch.maximum(ibp_margin_bounds(model, c, tau, norm, y), crown_ibp_margin_bounds(model, c, tau, norm, y))


def my_loss(model, x, y, t, tight=("linf", "l2")):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    ce = [F.cross_entropy((zhat_tight if p in tight else zhat)(model, x, y, eps_scale(t) * EPS[p], p), y,
                          reduction="none") for p in NORMS]
    return torch.maximum(*ce).mean()

In [ ]:
res_c = [results[3],  # cure_max, the best part (b) network
         run("ours (tight linf + l2)", my_loss, seeds=(0, 1)),
         run("ablation: tight l2 only", lambda m, x, y, t: my_loss(m, x, y, t, tight=("l2",)), seeds=(0, 1))]
# Same training time: train CURE-Max for more epochs.
ep = round(20 * res_c[1]["time"] / res_c[0]["time"])
res_c.append(run(f"cure_max, {ep} epochs", lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), epochs=ep))
print_results(res_c)

cure_max                  clean  92.19  linf  67.48  l2  57.20  union  57.07  pgd_linf  76.56  pgd_l2  78.08  pgd_union  75.97  time/epoch   3.6 s
ours (tight linf + l2)    clean  94.24  linf  71.31  l2  63.61  union  63.54  pgd_linf  77.81  pgd_l2  78.72  pgd_union  76.66  time/epoch  12.1 s
ablation: tight l2 only   clean  92.60  linf  71.55  l2  56.15  union  56.15  pgd_linf  76.81  pgd_l2  76.41  pgd_union  74.72  time/epoch   7.2 s
cure_max, 68 epochs       clean  92.51  linf  68.07  l2  57.93  union  57.86  pgd_linf  77.44  pgd_l2  78.61  pgd_union  76.86  time/epoch   3.8 s


**(c) Idea.** The verifier uses the tighter of the IBP and CROWN bounds, but CURE trains on IBP only. IBP is especially loose for $\ell_2$, since it turns the ball into a box after the first layer. We keep CURE-Max, but on each SABR small ball we train on the tighter of IBP and CROWN-IBP [Zhang et al., ICLR 2020]:
$$L = \frac{1}{n}\sum_i \max_{p} CE\big(-\max(\underline m^p_{IBP}, \underline m^p_{CROWN\text{-}IBP}),\ y_i\big)$$
New: Zhang et al. use CROWN-IBP for $\ell_\infty$ on the full ball. We use it for both norms inside CURE's small balls, taking the max of the two bounds like the verifier does.

**Process.** I first also tried combining the two norms with the CE of the elementwise worst-case logits. It didn't help (56.9%), so I kept only the bound change.

**Results.** +6.5 union (63.5%) and +2.1 clean (94.2%). Training with the same total time (CURE-Max for 68 epochs) gives only 57.9%, so the gain is not from extra compute.

**Ablation.** Tight bounds on $\ell_2$ only: union falls to 56.2%. The $\ell_2$ loss gets small, so the max almost always picks $\ell_\infty$ and $\ell_2$ barely trains.

**Analysis.** It works because the network is trained on a bound close to the verifier's. The certified-to-PGD gap shrinks from 20.9 to 15.1 points ($\ell_2$) and from 9.1 to 6.5 points ($\ell_\infty$). A tighter bound also regularizes less, so clean accuracy rises.